Install and Check Environment

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate sentencepiece

import torch
import transformers
import datasets
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nGPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.9 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
Transformers: 5.14.1
Datasets: 5.0.1
Scikit-learn: 1.6.1

GPU Available: True
GPU: Tesla T4


Upload and Inspect the DAPT Dataset

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df = pd.read_csv("dapt_dataset.csv")

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Saving dapt_dataset.csv to dapt_dataset.csv
Dataset loaded successfully!

Dataset Shape:
(12503, 2)

Columns:
['Sentences', 'Unnamed: 1']

First 5 rows:


,Sentences,Unnamed: 1
0,It would be better if a believing girl were th...,NaN
1,A flower never has to say what you smell like....,NaN
2,The son of a bitch is a magician and does magi...,NaN
3,Your hijras are fucking gay bitches.,NaN
4,Safa Kabir's cheek was hit with a shoe.,NaN


Clean the DAPT Dataset

In [ ]:
import pandas as pd

df = df[["Sentences"]].copy()

df = df.dropna(subset=["Sentences"])

df["Sentences"] = df["Sentences"].astype(str)

df["Sentences"] = df["Sentences"].str.strip()

df = df[df["Sentences"] != ""]

before = len(df)
df = df.drop_duplicates(subset=["Sentences"]).reset_index(drop=True)
after = len(df)

print("Cleaning completed!")

print(f"Rows before duplicate removal: {before}")
print(f"Rows after duplicate removal : {after}")
print(f"Duplicates removed           : {before - after}")

print("\nFinal shape:")
print(df.shape)

print("\nFirst 5 cleaned rows:")
display(df.head())

Cleaning completed!
Rows before duplicate removal: 12503
Rows after duplicate removal : 12143
Duplicates removed           : 360

Final shape:
(12143, 1)

First 5 cleaned rows:


,Sentences
0,It would be better if a believing girl were th...
1,A flower never has to say what you smell like....
2,The son of a bitch is a magician and does magi...
3,Your hijras are fucking gay bitches.
4,Safa Kabir's cheek was hit with a shoe.


In [ ]:
print([x for x in dir() if "df" in x.lower()])

['df']


Remove Usernames and URLs + Save Clean DAPT Dataset

In [ ]:
import re

dapt_clean = df.copy()


def clean_social_text(text):


    text = re.sub(r'@\w+', '', text)


    text = re.sub(
        r'http\S+|www\S+',
        '',
        text
    )


    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text


before = dapt_clean["Sentences"].copy()

dapt_clean["Sentences"] = dapt_clean["Sentences"].apply(
    clean_social_text
)



changed_rows = (
    before != dapt_clean["Sentences"]
).sum()


print("Username/URL cleaning completed!")
print("Rows modified:", changed_rows)



dapt_clean = dapt_clean[
    dapt_clean["Sentences"].str.len() > 0
]


print("\nFinal DAPT dataset shape:")
print(dapt_clean.shape)


print("\nFirst 5 cleaned rows:")
display(dapt_clean.head())


dapt_clean.to_csv(
    "dapt_dataset_clean.csv",
    index=False
)


print("\nSaved successfully: dapt_dataset_clean.csv")

Username/URL cleaning completed!
Rows modified: 1387

Final DAPT dataset shape:
(12143, 1)

First 5 cleaned rows:


,Sentences
0,It would be better if a believing girl were th...
1,A flower never has to say what you smell like....
2,The son of a bitch is a magician and does magi...
3,Your hijras are fucking gay bitches.
4,Safa Kabir's cheek was hit with a shoe.



Saved successfully: dapt_dataset_clean.csv


Load Clean Dataset for DAPT

In [ ]:
import pandas as pd
from datasets import Dataset


dapt_df = pd.read_csv(
    "dapt_dataset_clean.csv"
)


print("Dataset loaded!")
print("Shape:", dapt_df.shape)

print("\nColumns:")
print(dapt_df.columns.tolist())


dapt_dataset = Dataset.from_pandas(
    dapt_df
)


print("\nHugging Face dataset created!")
print(dapt_dataset)


print("\nExample:")
print(dapt_dataset[0])

Dataset loaded!
Shape: (12143, 1)

Columns:
['Sentences']

Hugging Face dataset created!
Dataset({
    features: ['Sentences'],
    num_rows: 12143
})

Example:
{'Sentences': "It would be better if a believing girl were the Prophet's sex slave, Amen."}


Load RoBERTa Tokenizer for DAPT

In [ ]:
from transformers import AutoTokenizer


model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


print("RoBERTa tokenizer loaded successfully!")
print("Model:", model_name)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

RoBERTa tokenizer loaded successfully!
Model: roberta-base
Vocabulary size: 50265
Maximum length: 512


Tokenize DAPT Dataset

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["Sentences"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


tokenized_dapt = dapt_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Sentences"]
)


print("DAPT tokenization completed!")

print(tokenized_dapt)

print("\nExample:")
print(tokenized_dapt[0])

Map:   0%|          | 0/12143 [00:00<?, ? examples/s]

DAPT tokenization completed!
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 12143
})

Example:
{'input_ids': [0, 243, 74, 28, 357, 114, 10, 13294, 1816, 58, 5, 28273, 18, 2099, 20174, 6, 36969, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


Prepare Masked Language Modeling (MLM)

In [ ]:
from transformers import DataCollatorForLanguageModeling


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)


print("MLM data collator created successfully!")
print("Mask probability: 15%")

MLM data collator created successfully!
Mask probability: 15%


Load Fresh RoBERTa MLM Model

In [ ]:
from transformers import AutoModelForMaskedLM


model_name = "roberta-base"


mlm_model = AutoModelForMaskedLM.from_pretrained(
    model_name
)


print("RoBERTa MLM model loaded successfully!")
print("Model:", model_name)

print("\nParameters:")
print(f"{mlm_model.num_parameters():,}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RoBERTa MLM model loaded successfully!
Model: roberta-base

Parameters:
124,697,433


Configure DAPT Training Arguments

In [ ]:
from transformers import TrainingArguments


training_args = TrainingArguments(

    output_dir="./roberta_dapt_checkpoints",

    num_train_epochs=3,

    per_device_train_batch_size=16,

    learning_rate=5e-5,

    weight_decay=0.01,

    fp16=True,

    logging_steps=100,

    save_strategy="epoch",

    report_to="none"
)


print("DAPT training arguments configured successfully!")

DAPT training arguments configured successfully!


Create DAPT Trainer

In [ ]:
from transformers import Trainer


trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=tokenized_dapt,
    data_collator=data_collator
)


print("DAPT Trainer created successfully!")

DAPT Trainer created successfully!


Train and Save RoBERTa DAPT Model

In [ ]:
dapt_training_output = trainer.train()


print("\nDAPT training completed successfully!")


dapt_model_path = "./roberta_dapt_model"

mlm_model.save_pretrained(dapt_model_path)
tokenizer.save_pretrained(dapt_model_path)


print("\nDAPT model saved successfully!")
print("Saved location:", dapt_model_path)


import os

print("\nSaved files:")
print(os.listdir(dapt_model_path))

Step,Training Loss
100,2.495918
200,2.424316
300,2.442666
400,2.398274
500,2.381817
600,2.342425
700,2.401869
800,2.285699
900,2.217411
1000,2.124343


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


DAPT training completed successfully!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


DAPT model saved successfully!
Saved location: ./roberta_dapt_model

Saved files:
['model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


Zip and Download DAPT Model

In [ ]:
import shutil
from google.colab import files


zip_path = shutil.make_archive(
    "roberta_dapt_model",
    "zip",
    "./roberta_dapt_model"
)


print("ZIP created successfully!")
print(zip_path)


files.download(zip_path)

ZIP created successfully!
/content/roberta_dapt_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>